In [2]:
# %%capture
# aaa
import time
import random
import torch
import optuna
import os
import re
import gc
import shap
import logging
import optuna

import pandas as pd
import numpy as np
import tensorflow as tf
import seaborn as sns
import xgboost as xgb
import matplotlib.pyplot as plt

from pandas.api.types import is_integer_dtype as is_integer
from pandas.api.types import is_float_dtype as is_float
from google.cloud import bigquery
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold, StratifiedKFold
from sklearn.feature_selection import RFECV
from sklearn.metrics import auc, average_precision_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, make_scorer, precision_recall_curve, PrecisionRecallDisplay, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

2026-02-06 15:20:45.253404: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Callable Functions

In [3]:
def calculate_metrics(baseline):
    # Probabilities of the positive class
    y_pred_test = baseline.predict_proba(X_test)[:, 1]
    y_pred_val = baseline.predict_proba(X_val)[:, 1]

    # Resetting the index of y_test for alignment
    y_test_reset = y_test.reset_index(drop=True)

    # ROC AUC Score
    roc_auc_test = roc_auc_score(y_test, y_pred_test)

    # Sorting indices by predicted probabilities in descending order
    idx = np.argsort(y_pred_test)[::-1]

    # Function to calculate metrics for a given cutoff percentile
    def calculate_percentile_metrics(y_test_reset, idx, percentile):
        cutoff = int(percentile * len(y_test_reset))  # Number of data points in the top percentile
        top_indices = idx[:cutoff]  # Indices of the top percentile predictions

        # Binary predictions for the top percentile
        predictions_binary = np.zeros(len(y_test_reset), dtype=int)
        predictions_binary[top_indices] = 1

        # Confusion matrix calculation
        tn, fp, fn, tp = confusion_matrix(y_test_reset, predictions_binary, labels=[0, 1]).ravel()

        # Performance metrics
        ppv = 100 * (tp / (tp + fp)) if (tp + fp) > 0 else 0  # Positive Predictive Value
        sensitivity = 100 * (tp / (tp + fn)) if (tp + fn) > 0 else 0  # Sensitivity or Recall

        # Lift calculation
        actual_positives_top_perc = y_test_reset[top_indices].sum()
        expected_positives = y_test_reset.sum() * percentile
        
        pos_rt_in_top_perc = y_test_reset[top_indices].mean()
        pos_rt_in_overall = y_test_reset.mean()
        
        #lift = actual_positives_top_perc / expected_positives if expected_positives > 0 else 0
        lift = pos_rt_in_top_perc / pos_rt_in_overall if pos_rt_in_overall > 0 else 0

        return lift, ppv, sensitivity

    lift_1_perc, ppv_1_perc, sensitivity_1_perc = calculate_percentile_metrics(y_test_reset, idx, 0.01)
    lift_10_perc, ppv_10_perc, sensitivity_10_perc = calculate_percentile_metrics(y_test_reset, idx, 0.10)

    return roc_auc_test, lift_1_perc, lift_10_perc, ppv_1_perc, sensitivity_1_perc, ppv_10_perc, sensitivity_10_perc

In [4]:
def check_label_distribution(y):
    # Count the number of occurrences of each label 
    count_0 = (y == 0).sum()
    count_1 = (y == 1).sum()
    ratio = count_1 / count_0 if count_0 != 0 else np.inf
    print(count_0, count_1, ratio)
    return count_0, count_1, ratio

In [5]:
def plot_lift_curve(y_val, y_pred, step=0.01):
    
    #Define an auxiliar dataframe to plot the curve
    aux_lift = pd.DataFrame()
    #Create a real and predicted column for our new DataFrame and assign values
    aux_lift['real'] = y_val
    aux_lift['predicted'] = y_pred
    #Order the values for the predicted probability column:
    aux_lift.sort_values('predicted',ascending=False,inplace=True)
    
    #Create the values that will go into the X axis of our plot
    x_val = np.arange(step,1+step,step)
    #Calculate the ratio of ones in our data
    ratio_ones = aux_lift['real'].sum() / len(aux_lift)
    #Create an empty vector with the values that will go on the Y axis our our plot
    y_v = []
    
    #Calculate for each x value its correspondent y value
    for x in x_val:
        num_data = int(np.ceil(x*len(aux_lift))) #The ceil function returns the closest integer bigger than our number 
        data_here = aux_lift.iloc[:num_data,:]   # ie. np.ceil(1.4) = 2
        ratio_ones_here = data_here['real'].sum()/len(data_here)
        y_v.append(ratio_ones_here / ratio_ones)
           
   #Plot the figure
    fig, axis = plt.subplots()
    fig.figsize = (40,40)
    axis.plot(x_val, y_v, 'g-', linewidth = 3, markersize = 5)
    axis.plot(x_val, np.ones(len(x_val)), 'k-')
    axis.set_xlabel('Proportion of sample')
    axis.set_ylabel('Lift')
    plt.title('Lift Curve')
    plt.show()

In [6]:
def lift_chart(X,actual_target,model):
    avg_tgt = actual_target.sum()/len(actual_target)
    df_data = X.copy()
    X_data = df_data.copy()
    df_data['Actual'] = actual_target
    df_data['Predict']= model.predict(X_data)
    y_Prob = pd.DataFrame(model.predict_proba(X_data))
    df_data['Prob_1']=list(y_Prob[1])
    df_data.sort_values(by=['Prob_1'],ascending=False,inplace=True)
    df_data.reset_index(drop=True,inplace=True)
    df_data['Percentile']=pd.qcut(df_data.index,100,labels=False)
    output_df = pd.DataFrame()
    grouped = df_data.groupby('Percentile',as_index=False)
    output_df['Max_Scr']=grouped.max().Prob_1
    output_df['Min_Scr']=grouped.min().Prob_1
    output_df['Actual']=grouped.sum().Actual
    output_df['Total']=grouped.count().Actual
    output_df["Population_perc"] = (output_df["Total"]/len(actual_target))*100
    output_df['Per_Events'] = (output_df['Actual']/output_df['Actual'].sum())*100
    output_df['Gain_Percentage'] = output_df.Per_Events.cumsum()
    output_df["Cumulative_Population"] = output_df.Population_perc.cumsum()
    output_df["Lift"] = output_df["Gain_Percentage"]/output_df["Cumulative_Population"]
    return output_df

In [7]:
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

client = bigquery.Client()

### Parameter setting for script

In [8]:
target_names = ['without outcome', 'with outcome', ] # used to pass labels to confusion matrices in metrics section

threshold = 0.25 # custom decision point threshold if you don't want to go with default 0.5

model_name = "cancer_IP_XGBoost_v_case" # Model name for use in plots

one_hot_backup = "cancer_xgboost_preprocess_v_case.feather" #back up computationally intensive step so it only needs to be completed once

outcome_var = "cancer_case" #name the variable that we are training to predict so we can pull this column as our y automatically anywhere pertinent

random_st = 53 # set a seed for replicability

feature_vars = [1, 2, 4] + [i for i in range(13, 588)] + [i for i in range(589, 594)] # column indices for the features you want to be used for creating the test/train/val X matrices

indexing_var = 'individual_id' #name of the column to index on (i.e. individual_id)

#vars for undersampling - can add or remove values based on your sample's specific ratio of rare outcome
ratios = [0.015, 0.02, 0.03, 0.05]#, 0.075, 0.1]#[0.03, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]  # Undersampling ratios
seeds = [53]  # Random seeds for reproducibility

save_model_nm = 'cancer_xgboost_v_case_mlops.cbm' #name you want the model to be saved under

### Load Data

In [9]:
# Get outcome
sql = """
SELECT
individual_id,
Oncologic_case_post as cancer_case
    FROM `anbc-hcb-dev.clin_analytics_hcb_dev.a367726_ss_revamp_training_all_cases_analytic_w_model`
WHERE train_test_split <= 75
"""
outcome = client.query(sql).to_dataframe() 

outcome.shape

(1045104, 2)

In [10]:
outcome.head()

,individual_id,cancer_case
0,832863257,0
1,7368464822586842,0
2,416252484273,0
3,6958235640524996,0
4,596272447424,0


In [11]:
# Get features
sql = """
SELECT
* except(Cardiac_case_post, Digestive_case_post, Endocrine_Metabolic_case_post, Infectious_case_post,
Injury_Poisoning_case_post, Musculoskeletal_case_post, Neurologic_case_post, Obstetric_case_post,
Oncologic_case_post, Renal_case_post, Respiratory_case_post,
Cardiac_case_post_1_7, Digestive_case_post_1_7, Endocrine_Metabolic_case_post_1_7, Infectious_case_post_1_7,
Injury_Poisoning_case_post_1_7, Musculoskeletal_case_post_1_7,
Neurologic_case_post_1_7, Obstetric_case_post_1_7, 
Oncologic_case_post_1_7, Renal_case_post_1_7, Respiratory_case_post_1_7)
FROM `anbc-hcb-dev.clin_analytics_hcb_dev.a367726_ss_revamp_training_all_cases_analytic_w_model`
WHERE train_test_split <= 75
"""
features = client.query(sql).to_dataframe() 
sql2 = """
SELECT
* except(Cardiac_case_post, Digestive_case_post, Endocrine_Metabolic_case_post, Infectious_case_post,
Injury_Poisoning_case_post, Musculoskeletal_case_post, Neurologic_case_post, Obstetric_case_post,
Oncologic_case_post, Renal_case_post, Respiratory_case_post,
Cardiac_case_post_1_7, Digestive_case_post_1_7, Endocrine_Metabolic_case_post_1_7, Infectious_case_post_1_7,
Injury_Poisoning_case_post_1_7, Musculoskeletal_case_post_1_7,
Neurologic_case_post_1_7, Obstetric_case_post_1_7, 
Oncologic_case_post_1_7, Renal_case_post_1_7, Respiratory_case_post_1_7)
FROM `anbc-hcb-dev.clin_analytics_hcb_dev.a367726_ss_revamp_training_all_cases_analytic_w_model`
WHERE train_test_split > 75
"""
final_val_features = client.query(sql2).to_dataframe() 
features.shape

(1045104, 590)

In [15]:
#set null embeddings to 0 (standard protocol)
emb_pattern = r'emb[0-255]+'
emb_col = [col for col in features.columns if re.match(emb_pattern ,col)]
features[emb_col] = features[emb_col].fillna(0)
for c in features.columns: 
    dt = features[c].dtype
    if is_integer(dt) or is_float(dt):
        features[c] = features[c].fillna(0) 
        # print("Floatint:", dt)
    else:
        try:
            features[c] = features[c].fillna('')
        except:
            print("ERROR - DATE VARIABLE FOUND", dt)
    dt = final_val_features[c].dtype
    if is_integer(dt) or is_float(dt):
        final_val_features[c] = final_val_features[c].fillna(0) 
        # print("Floatint:", dt)
    else:
        try:
            final_val_features[c] = final_val_features[c].fillna('')
        except:
            print("ERROR - DATE VARIABLE FOUND", dt)

ERROR - DATE VARIABLE FOUND dbdate
ERROR - DATE VARIABLE FOUND dbdate


### One hot encode (if needed)

In [17]:
#  0 := categorical, 1 := continuous, 2 := binary
# example variables
nem_to_type = {
    'drug_ind':0, 
    #'urbsubr':0, 
    'gender_cd':0, # TODO: IF = M, 1 ELIF = F 0 ELSE NULL 
    'region':0
    #'tenure_yr1':1,
}

In [18]:
index_to_feature = dict(enumerate(features.columns))
feature_to_index = {value: key for key, value in index_to_feature.items()}
categorical_features = [feature for feature in nem_to_type if nem_to_type[feature] == 0]
categorical_indices = [feature_to_index[feature] for feature in categorical_features if feature in feature_to_index]
len(categorical_features)

index_to_feature_fv = dict(enumerate(final_val_features.columns))
feature_to_index_fv = {value: key for key, value in index_to_feature_fv.items()}
categorical_features_fv = [feature for feature in nem_to_type if nem_to_type[feature] == 0]
categorical_indices_fv = [feature_to_index_fv[feature] for feature in categorical_features_fv if feature in feature_to_index_fv]


In [19]:
#features.groupby('region').count()
#features.columns

In [20]:
#categorical_features.remove('index_dt')
features[categorical_features] = features[categorical_features].astype(str)
features['gender_cd'] = features['gender_cd'].map({'M': 1, 'F': 0}).fillna(-1)  
categorical_features.remove('gender_cd')
features['drug_ind'] = features['drug_ind'].map({'Y': 1, 'N': 0}).fillna(-1) 
categorical_features.remove('region')
#features['region'] = features['region'].map({'missing': 0, 'regNE': 1, 'regMW': 2, 'regS': 3, 'regW': 4}).fillna(-1) 
features_clean = pd.get_dummies(features, columns=['region'])



#final_val_features[categorical_features_fv] = features[categorical_features_fv].astype(str)
final_val_features['gender_cd'] = final_val_features['gender_cd'].map({'M': 1, 'F': 0}).fillna(-1)  
#categorical_features_fv.remove('gender_cd')
final_val_features['drug_ind'] = final_val_features['drug_ind'].map({'Y': 1, 'N': 0}).fillna(-1) 
#final_val_features['region'] = final_val_features['region'].map({'missing': 0, 'regNE': 1, 'regMW': 2, 'regS': 3, 'regW': 4}).fillna(-1) 
final_val_features_clean = pd.get_dummies(final_val_features, columns=['region'])


In [24]:
#Set index to member ID for easy down-stream tracking
features_clean.set_index(indexing_var, inplace = True)
final_val_features_clean.set_index(indexing_var, inplace = True)
outcome.set_index(indexing_var, inplace = True)
df = outcome.merge(features_clean, on=indexing_var, how='inner')

In [26]:
df.to_feather(one_hot_backup)

### Create test/train/validation split

In [27]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, feature_vars], df.loc[:, outcome_var], test_size = 0.2, random_state = random_st, stratify = df.loc[:, outcome_var])
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size = 0.5, random_state = random_st, stratify = y_test)

In [29]:
X_train_save = X_train
X_val_save = X_val
X_test_save = X_test

In [30]:
X_train.index_month

individual_id
97833906299357      4
422798041451162     4
918978197           6
62061966239503     10
15711              10
                   ..
61514941398         6
13148690067766      4
5170282900878       9
89331718901792      6
74605871492347      2
Name: index_month, Length: 836083, dtype: Int64

In [32]:
y_train = y_train.astype('int')
y_val = y_val.astype('int')
y_test = y_test.astype('int')

In [33]:
import pandas as pd
# open file in read mode
f = open('xgboost_selected_features_cancer_v_case.txt', 'r')
selected_features = [x.strip() for x in f.readlines()]
print(len(selected_features))
f.close()

X_train_df = pd.DataFrame(X_train, columns = list(set(list(df.columns) + ['rando'])))

X_train = X_train_df[selected_features]

X_val_df = pd.DataFrame(X_val, columns = list(set(list(df.columns) + ['rando'])))

X_val = X_val_df[selected_features]

X_test_df = pd.DataFrame(X_test, columns = list(set(list(df.columns) + ['rando'])))

X_test = X_test_df[selected_features]

# X_test

156


In [34]:
print(xgb.__version__)

X_train_use = xgb.DMatrix(X_train, label=y_train)
X_test_use = xgb.DMatrix(X_test, label=y_test)
X_val_use = xgb.DMatrix(X_val, label=y_val)

2.1.4


### Build optimal model

In [75]:
# Optimal model
import xgboost as xgb
#params = {
#    'objective': 'binary:logistic',
#    'random_state': 53, 
#    'tree_method' : "hist", 
#    'device' : "cuda",
#    'n_jobs': -1,
#    'learning_rate': 0.008410840153403855, 
#    'num_boost_round': 3101, #'n_estimators': 3101,
#    'max_depth': 4, 
#    'subsample': 0.5826667789672917, 
#    'colsample_bytree': 0.551691605418298
#}

#params = {
#        'objective': 'binary:logistic',
#        'tree_method': 'hist',
#        'grow_policy': 'depthwise',
#        
#        # Hyperparameters (tuned)
#        'eta': 0.008410840153403855,
#        'max_depth': 4,
#        'subsample': 0.5826667789672917, 
#        'colsample_bytree': 0.551691605418298,
#        'num_boost_round': 3101,
#        'min_child_weight': 1,
#        'reg_lambda': 1,
#        'scale_pos_weight': 1, 
#    
#        # Fixed parameters
#        'random_state': 53,  # For reproducibility
#        'verbosity': 1,
#        'eval_metric': 'logloss'
#    }

params = {'objective': 'binary:logistic',
 'base_score': 0.5,
 'booster': 'gbtree',
 'colsample_bylevel': 1,
 'colsample_bynode': 1,
 'colsample_bytree': 0.551691605418298,
 'device': 'cuda',
 'eval_metric': 'logloss',
 'gamma': 0,
 'grow_policy': 'lossguide',
 #'interaction_constraints': None,
 'learning_rate': 0.008410840153403855,
 'max_bin': 256,
 #'max_cat_threshold': None,
 #'max_cat_to_onehot': None,
 'max_delta_step': 0,
 'max_depth': 4,
 'max_leaves': 0,
 'min_child_weight': 1,
 #'monotone_constraints': None,
 'multi_strategy': 'one_output_per_tree',
 'n_jobs': -1,
 'num_parallel_tree': 1,
 'random_state': 53,
 'reg_alpha': 0,
 'reg_lambda': 1,
 'sampling_method': 'uniform',
 'scale_pos_weight': 1,
 'subsample': 0.5826667789672917,
 'tree_method': 'hist',
 'validate_parameters': None,
 'verbosity': 1}

#model = xgb.XGBClassifier(**params)
#model.train(X_train, y_train, eval_set=[(X_val, y_val)], verbose=0)

model = xgb.train(
        params=params,
        dtrain=X_train_use,
        num_boost_round=3101,#args.num_boost_round,
        evals=[(X_val_use, 'eval')], #(X_train_use, 'train'), 
        #early_stopping_rounds=500,
        verbose_eval=100
    )

[0]	eval-logloss:0.68506


/opt/conda/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [16:40:04] WARNING: /workspace/src/context.cc:43: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [16:40:04] WARNING: /workspace/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


[100]	eval-logloss:0.25620
[200]	eval-logloss:0.12017
[300]	eval-logloss:0.06905
[400]	eval-logloss:0.04935
[500]	eval-logloss:0.04187
[600]	eval-logloss:0.03910
[700]	eval-logloss:0.03804
[800]	eval-logloss:0.03756
[900]	eval-logloss:0.03732
[1000]	eval-logloss:0.03716
[1100]	eval-logloss:0.03704
[1200]	eval-logloss:0.03695
[1300]	eval-logloss:0.03688
[1400]	eval-logloss:0.03684
[1500]	eval-logloss:0.03680
[1600]	eval-logloss:0.03676
[1700]	eval-logloss:0.03672
[1800]	eval-logloss:0.03670
[1900]	eval-logloss:0.03667
[2000]	eval-logloss:0.03666
[2100]	eval-logloss:0.03664
[2200]	eval-logloss:0.03662
[2300]	eval-logloss:0.03661
[2400]	eval-logloss:0.03659
[2500]	eval-logloss:0.03658
[2600]	eval-logloss:0.03657
[2700]	eval-logloss:0.03655
[2800]	eval-logloss:0.03655
[2900]	eval-logloss:0.03654
[3000]	eval-logloss:0.03654
[3100]	eval-logloss:0.03653


In [76]:
model.save_model(save_model_nm)

/opt/conda/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [16:41:46] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


In [77]:
## predict for holdout set
#print(selected_features)

fv_only = final_val_features_clean[selected_features]
x_fv = xgb.DMatrix(fv_only)###pd.DataFrame(final_val_features_clean)#, columns = df.columns)

#fv_only = fv_only.drop('index_dt', axis=1)
#x_fv.head()




#x_fv = x_fv[X_test.columns]

#print(model.predict(X_test_use))

#final_preds_to_upload = model.predict(x_fv[X_test.columns])
final_preds_to_upload = model.predict(x_fv)
#fv_only['individual_id'] = fv_only.index

In [78]:
from google.cloud import bigquery
from google.auth import impersonated_credentials

job_labels = {"costcenter": 13070, "owner": "parked_aetna_com"}
load_config = bigquery.LoadJobConfig(labels=job_labels)

### NOTE: change below to the table used in production ###
TARGET_TABLE =""

            
#y_pred_test_class = model.predict(df)    
final_val_features_clean['individual_id'] = final_val_features_clean.index
final_val_features_clean['y_pred_test'] = final_preds_to_upload#[:,1]

output = final_val_features_clean[['individual_id', 'index_dt', 'y_pred_test']]
#output = output.reset_index()

load_job = client.load_table_from_dataframe(
    dataframe = output,
    destination = TARGET_TABLE,
    job_config = load_config,
)

print('donezo')


donezo
